class PlayerProfile:
    def __init__(self, name, team, position, overall_rating):
        self.name = name
        self.team = team
        self.position = position.lower()
        self.overall_rating = overall_rating
        self.base_rating = 6
        self.match_rating = self.base_rating

    def adjust_rating(self, delta):
        self.match_rating = max(0, min(10, self.match_rating + delta))

    def get_rating_boost(self):
        if self.overall_rating >= 87:
            return 0.2
        elif self.overall_rating >= 84:
            return 0.25
        elif self.overall_rating >= 81:
            return 0.3
        elif self.overall_rating >= 78:
            return 0.35
        elif self.overall_rating >= 75:
            return 0.40
        else:
            return 0.45

    def is_forward(self): return self.position == "forward"
    def is_midfielder(self): return self.position == "midfielder"
    def is_defender(self): return self.position == "defender"

    def input_match_stats(self, minutes=0, goals=0, assists=0,
                          yellow_card=False, red_card=False, clean_sheet=False, goals_conceded=0,
                          total_shots=0, shots_on_target=0, shots_off_target=0, tackles_loss=0,
                          total_passes=0, accurate_passes=0,
                          expected_goals=0, expected_assists=0,
                          big_chances_missed=0,
                          successful_dribbles=0, total_dribbles=0, conceded_penalty=0, missed_penalty=0,
                          accurate_crosses=0, total_crosses=0,
                          accurate_long_balls=0, total_long_balls=0,
                          dispossessed=0, tackles_won=0, interceptions=0,
                          clearances=0, ball_recoveries=0,
                          dribbled_past=0, duels_won=0, duels_lost=0,
                          ground_duels_won=0, ground_duels_total=0,
                          aerial_duels_won=0, aerial_duels_total=0, own_goal=0,
                          fouled=0, number_of_fouls=0):

        boost = self.get_rating_boost()

        # Goals and Assists
        self.adjust_rating(goals * 1)
        self.adjust_rating(assists * 1)

        if goals_conceded > 2:
            self.adjust_rating(-1)
            

        
        # Cards
        if yellow_card:
            self.adjust_rating(-1)
        if red_card:
            self.adjust_rating(-2)
                #Clean Sheet Bonus
        if clean_sheet:
            self.adjust_rating(boost)

        # Shot Accuracy
        if total_shots > 0:
            accuracy = shots_on_target / total_shots
            if accuracy >= 0.7:
                self.adjust_rating(boost)
            shots_off_target = total_shots - shots_on_target
            off_target_ratio = shots_off_target / total_shots
            if off_target_ratio >= 0.75:
                self.adjust_rating(-0.3)
            elif off_target_ratio >= 0.5:
                self.adjust_rating(-0.2)
            elif off_target_ratio >= 0.3:
                self.adjust_rating(-0.1)

        # xG comparison
        if goals > expected_goals:
            self.adjust_rating(boost)
        elif expected_goals > goals:
            self.adjust_rating(-0.2)

        # xA comparison
        if assists > expected_assists:
            self.adjust_rating(boost)
        elif expected_assists > assists:
            self.adjust_rating(-0.2)

        # Passing
        if total_passes > 0:
            pass_accuracy = accurate_passes / total_passes
            if pass_accuracy >= 0.9:
                self.adjust_rating(boost)
            elif pass_accuracy >= 0.75:
                self.adjust_rating(boost * 0.75)
            elif pass_accuracy >= 0.5:
                self.adjust_rating(boost * 0.5)
            else:
                self.adjust_rating(-0.3)

        # Dribbling Success Rate
        if total_dribbles > 0:
            dribble_rate = successful_dribbles / total_dribbles
            if dribble_rate >= 0.8:
                self.adjust_rating(boost)
            elif dribble_rate >= 0.5:
                self.adjust_rating(boost * 0.5)
            else:
                self.adjust_rating(-0.2)
            if tackles_won > tackles_loss:
                self.adjust_rating(boost)
            elif tackles_won < tackles_loss:
                self.adjust_rating(-boost)
             
        # Crossing
        if total_crosses > 0:
            crossing_rate = accurate_crosses / total_crosses
            if crossing_rate >= 0.5:
                self.adjust_rating(boost)
            elif crossing_rate >= 0.3:
                self.adjust_rating(boost * 0.5)

        # Long Balls
        if total_long_balls > 0:
            long_ball_rate = accurate_long_balls / total_long_balls
            if long_ball_rate >= 0.5:
                self.adjust_rating(boost)
            elif long_ball_rate >= 0.3:
                self.adjust_rating(boost * 0.5)

        # Possession Loss
        if dispossessed >= 3:
            self.adjust_rating(-0.1 * dispossessed)

        # Defensive Metrics
        if tackles_won >= 2:
            self.adjust_rating(tackles_won * 0.1)
        if interceptions >= 2:
            self.adjust_rating(interceptions * 0.1)
        if clearances >= 2:
            self.adjust_rating(clearances * 0.05)
        if ball_recoveries >= 5:
            self.adjust_rating(ball_recoveries * 0.05)
        if dribbled_past >= 2:
            self.adjust_rating(-0.1 * dribbled_past)

        # Duels
        total_duels = duels_won + duels_lost
        if total_duels > 0:
            duel_win_rate = duels_won / total_duels
            if duel_win_rate >= 0.6:
                self.adjust_rating(boost)
            elif duel_win_rate < 0.4:
                self.adjust_rating(-boost)

        if ground_duels_total > 0:
            ground_rate = ground_duels_won / ground_duels_total
            if ground_rate >= 0.6:
                self.adjust_rating(boost * 0.5)

        if aerial_duels_total > 0:
            aerial_rate = aerial_duels_won / aerial_duels_total
            if aerial_rate >= 0.6:
                self.adjust_rating(boost * 0.5)

        # Fouls and Being Fouled
        self.adjust_rating(fouled * 0.05)
        self.adjust_rating(-number_of_fouls * 0.1)

        # Played Full 90 Minutes
        if minutes >= 90:
            self.adjust_rating(0.2)

        # Big Chances Missed
        if big_chances_missed > 0:
            self.adjust_rating(-0.3 * big_chances_missed)

        if conceded_penalty > 0:
            self.adjust_rating(-2)
        if missed_penalty > 0:
            self.adjust_rating(-2)
        if minutes >= 90:
            self.adjust_rating(0.2) 
        if own_goal > 0:
            self.adjust_rating(-2)

class GoalkeeperProfile:
    def __init__(self, name, team, position, overall_rating):
        self.name = name
        self.team = team
        self.position = position.lower()
        self.overall_rating = overall_rating
        self.base_rating = 6
        self.match_rating = self.base_rating

    def adjust_rating(self, delta):
        self.match_rating = max(0, min(10, self.match_rating + delta))

    def get_rating_boost(self):
        if self.overall_rating >= 87:
            return 0.2
        elif self.overall_rating >= 84:
            return 0.25
        elif self.overall_rating >= 81:
            return 0.3
        elif self.overall_rating >= 78:
            return 0.35
        elif self.overall_rating >= 75:
            return 0.4
        else:
            return 0.5

    def input_match_stats(self,
                          saves=0,
                          goals_conceded=0,
                          xG_faced=0,
                          saves_inside_box=0,
                          saves_outside_box=0,
                          touches=0,
                          goals_prevented=0,
                          errors=0,
                          minutes_played=0,
                          penalty_saves=0,
                          total_passes=0,
                          accurate_passes=0,
                          total_long_balls=0,
                          accurate_long_balls=0,
                          yellow_card=False,
                          red_card=False,
                          clean_sheet=False):

        boost = self.get_rating_boost()

        # Saves
        self.adjust_rating(saves * 0.1)
        if saves_inside_box > 0:
            self.adjust_rating(saves_inside_box * 0.15)
        if saves_outside_box > 0:
            self.adjust_rating(saves_outside_box * 0.1)

        # Goals Conceded vs xG
        if goals_conceded < xG_faced:
            self.adjust_rating(boost)
        elif goals_conceded > xG_faced:
            self.adjust_rating(-boost)

                # Passing
        if total_passes > 0:
            pass_accuracy = accurate_passes / total_passes
            if pass_accuracy >= 0.9:
                self.adjust_rating(boost)
            elif pass_accuracy >= 0.75:
                self.adjust_rating(boost * 0.75)
            elif pass_accuracy >= 0.5:
                self.adjust_rating(boost * 0.5)
            else:
                self.adjust_rating(-0.3)

        # Goals prevented
        if goals_prevented > 0:
            self.adjust_rating(goals_prevented * 0.2)

        # Errors
        if errors == 0:
            self.adjust_rating(boost * 0.5)
        elif errors > 0:
            self.adjust_rating(-0.5 * errors)

        # Touches which can lead to distribution
        if touches >= 40:
            self.adjust_rating(boost * 0.5)
        elif touches >= 25:
            self.adjust_rating(boost * 0.2)

        # Penalty saves
        if penalty_saves > 0:
            self.adjust_rating(penalty_saves * 0.8)

        # Clean sheet 
        if clean_sheet and minutes_played >= 70:
            self.adjust_rating(0.5)

        # Played full match
        if minutes_played >= 90:
            self.adjust_rating(0.2)

        
        # Cards
        if yellow_card:
            self.adjust_rating(-1)
        if red_card:
            self.adjust_rating(-2)


In [60]:
#Home Team

home_world_cup_starting_xi_vs_away = [
    ["Jordan Pickford", "GK", "Everton", 83],
    ["Nico O'Reilly", "LB", "Manchester City", 81],
    ["Reece James", "RB", "Chelsea", 85],
    ["John Stones", "CB", "Manchester City", 81],
    ["Ezri Konsa", "CB", "Aston Villa", 83],
    ["Elliot Anderson", "CM", "Nottingham Forest", 84],    
    ["Declan Rice", "CDM", "Arsenal", 88],
    ["Jude Bellingham", "CM", "Real Madrid", 89],
    ["Anthony Gordon", "LW", "Barcelona", 81],
    ["Noni Madueke", "RW", "Arsenal", 79],
    ["Harry Kane", "ST", "Bayern Munich", 92],
]

 
hometeam_world_cup_bench_vs_awayteam= [
    ["Marcus Rashford", "LW", "Manchester United", 85],
    ["Bukayo Saka", "RW", "Arsenal", 87],
    ["Morgan Rogers", "CM", "Aston Villa", 83],
    ["Marc Guehi", "CB", "Manchester City", 85],
    ["Djed Spence", "RB", "Tottenham", 79],

]


In [80]:
gk = GoalkeeperProfile("Pickford", "England", "goalkeeper", 83)
gk.input_match_stats(
    minutes_played=90,
    saves=3,
    saves_inside_box=1,
    saves_outside_box=2,
    goals_conceded=2,
    xG_faced=0.51,
    goals_prevented=-1.49,
    total_passes=55,
    accurate_passes=43,
    total_long_balls=19,
    accurate_long_balls=7,
    touches=72,
    errors=1,
    penalty_saves=0,
    clean_sheet=False,
    yellow_card=False
)
print(f"{gk.name}'s match rating: {gk.match_rating:.2f}")

Pickford's match rating: 6.43


In [82]:
player = PlayerProfile("James", "England", "defender", 85)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=0,
    total_shots=1,
    shots_on_target=0,
    accurate_passes=36,
    total_passes=43,
    expected_goals=0.04,
    expected_assists=0.01,
    successful_dribbles=0,
    total_dribbles=1,
    accurate_crosses=0,
    total_crosses=1,
    accurate_long_balls=0,
    total_long_balls=3,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=3,
    interceptions=1,
    ball_recoveries=1,
    dribbled_past=0,
    duels_won=2,
    duels_lost=5,
    ground_duels_won=0,
    ground_duels_total=2,
    aerial_duels_won=2,
    aerial_duels_total=5,
    fouled=0,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=2 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

James's match rating: 5.49


In [84]:
player = PlayerProfile("Stones", "England", "defender", 81)

player.input_match_stats(
    minutes=87,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=63,
    total_passes=65,
    expected_goals=0,
    expected_assists=0.01,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=2,
    total_long_balls=4,
    dispossessed=0,
    tackles_won=1,
    tackles_loss=0,
    clearances=1,
    interceptions=0,
    ball_recoveries=5,
    dribbled_past=1,
    duels_won=4,
    duels_lost=3,
    ground_duels_won=2,
    ground_duels_total=4,
    aerial_duels_won=2,
    aerial_duels_total=3,
    fouled=1,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=2 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Stones's match rating: 6.75


In [86]:
player = PlayerProfile("Konsa", "England", "defender", 83)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=0,
    big_chances_missed=1,
    total_shots=2,
    shots_on_target=1,
    accurate_passes=67,
    total_passes=71,
    expected_goals=0.63,
    expected_assists=0,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=3,
    dribbled_past=0,
    duels_won=3,
    duels_lost=5,
    ground_duels_won=2,
    ground_duels_total=3,
    aerial_duels_won=1,
    aerial_duels_total=5,
    fouled=2,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=2 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Konsa's match rating: 5.85


In [88]:
player = PlayerProfile("O'Reilly", "England", "defender", 81)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=0,
    total_shots=2,
    shots_on_target=1,
    accurate_passes=36,
    total_passes=43,
    expected_goals=0.18,
    expected_assists=0.06,
    successful_dribbles=3,
    total_dribbles=3,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=1,
    total_long_balls=2,
    dispossessed=1,
    tackles_won=0,
    tackles_loss=0,
    clearances=1,
    interceptions=0,
    ball_recoveries=1,
    dribbled_past=2,
    duels_won=5,
    duels_lost=5,
    ground_duels_won=4,
    ground_duels_total=8,
    aerial_duels_won=1,
    aerial_duels_total=2,
    fouled=1,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=2 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

O'Reilly's match rating: 6.37


In [90]:
player = PlayerProfile("Anderson", "England", "midfielder", 84)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=1,
    total_shots=1,
    shots_on_target=0,
    accurate_passes=47,
    total_passes=55,
    expected_goals=0.04,
    expected_assists=0.02,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=1,
    total_crosses=1,
    accurate_long_balls=2,
    total_long_balls=3,
    dispossessed=0,
    tackles_won=2,
    tackles_loss=0,
    clearances=2,
    interceptions=4,
    ball_recoveries=8,
    dribbled_past=0,
    duels_won=6,
    duels_lost=5,
    ground_duels_won=4,
    ground_duels_total=7,
    aerial_duels_won=2,
    aerial_duels_total=4,
    fouled=2,
    number_of_fouls=3,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Anderson's match rating: 8.74


In [92]:
player = PlayerProfile("Rice", "England", "midfielder", 88)

player.input_match_stats(
    minutes=72,
    goals=0,
    assists=1,
    total_shots=1,
    shots_on_target=1,
    accurate_passes=32,
    total_passes=37,
    expected_goals=0.04,
    expected_assists=0.22,
    successful_dribbles=1,
    total_dribbles=1,
    accurate_crosses=4,
    total_crosses=9,
    accurate_long_balls=0,
    total_long_balls=1,
    dispossessed=1,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=4,
    dribbled_past=0,
    duels_won=2,
    duels_lost=2,
    ground_duels_won=1,
    ground_duels_total=3,
    aerial_duels_won=1,
    aerial_duels_total=1,
    fouled=0,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Rice's match rating: 7.65


In [94]:
player = PlayerProfile("Bellingham", "England", "midfielder", 89)

player.input_match_stats(
    minutes=80,
    goals=1,
    big_chances_missed=1,
    assists=0,
    total_shots=3,
    shots_on_target=2,
    accurate_passes=18,
    total_passes=24,
    expected_goals=0.48,
    expected_assists=0,
    successful_dribbles=1,
    total_dribbles=1,
    accurate_crosses=0,
    total_crosses=1,
    accurate_long_balls=0,
    total_long_balls=1,
    dispossessed=1,
    tackles_won=3,
    tackles_loss=0,
    clearances=1,
    interceptions=1,
    ball_recoveries=5,
    dribbled_past=0,
    duels_won=5,
    duels_lost=2,
    ground_duels_won=5,
    ground_duels_total=6,
    aerial_duels_won=0,
    aerial_duels_total=1,
    fouled=1,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Bellingham's match rating: 8.25


In [96]:
player = PlayerProfile("Gordon", "England", "forward", 81)

player.input_match_stats(
    minutes=72,
    goals=0,
    assists=0,
    total_shots=2,
    shots_on_target=1,
    accurate_passes=9,
    total_passes=11,
    expected_goals=0.27,
    expected_assists=0.01,
    successful_dribbles=0,
    total_dribbles=2,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=1,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=0,
    dribbled_past=0,
    duels_won=0,
    duels_lost=2,
    ground_duels_won=0,
    ground_duels_total=2,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Gordon's match rating: 5.12


In [98]:
player = PlayerProfile("Kane", "England", "forward", 92)

player.input_match_stats(
    minutes=90,
    goals=2,
    assists=0,
    total_shots=6,
    shots_on_target=3,
    accurate_passes=14,
    total_passes=21,
    expected_goals=1.03,
    expected_assists=0.04,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=1,
    total_long_balls=3,
    dispossessed=1,
    tackles_won=1,
    tackles_loss=0,
    clearances=1,
    interceptions=0,
    ball_recoveries=1,
    dribbled_past=0,
    duels_won=4,
    duels_lost=5,
    ground_duels_won=2,
    ground_duels_total=5,
    aerial_duels_won=2,
    aerial_duels_total=4,
    fouled=1,
    number_of_fouls=2,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Kane's match rating: 8.25


In [100]:
player = PlayerProfile("Madueke", "England", "forward", 79)

player.input_match_stats(
    minutes=72,
    goals=0,
    assists=0,
    total_shots=1,
    shots_on_target=0,
    accurate_passes=17,
    total_passes=18,
    expected_goals=0.08,
    expected_assists=0.41,
    successful_dribbles=1,
    total_dribbles=1,
    accurate_crosses=1,
    total_crosses=1,
    accurate_long_balls=1,
    total_long_balls=1,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=1,
    ball_recoveries=1,
    dribbled_past=1,
    duels_won=2,
    duels_lost=2,
    ground_duels_won=2,
    ground_duels_total=3,
    aerial_duels_won=0,
    aerial_duels_total=1,
    fouled=1,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Madueke's match rating: 6.92


In [102]:
player = PlayerProfile("Rashford", "England", "forward", 85)

player.input_match_stats(
    minutes=18,
    goals=1,
    assists=0,
    total_shots=1,
    shots_on_target=1,
    accurate_passes=3,
    total_passes=5,
    expected_goals=0.06,
    expected_assists=0,
    successful_dribbles=0,
    total_dribbles=3,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=1,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=1,
    interceptions=0,
    ball_recoveries=2,
    dribbled_past=1,
    duels_won=0,
    duels_lost=5,
    ground_duels_won=0,
    ground_duels_total=4,
    aerial_duels_won=0,
    aerial_duels_total=1,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Rashford's match rating: 7.17


In [104]:
player = PlayerProfile("Saka", "England", "forward", 87)

player.input_match_stats(
    minutes=18,
    goals=0,
    assists=1,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=5,
    total_passes=6,
    expected_goals=0,
    expected_assists=0.01,
    successful_dribbles=1,
    total_dribbles=1,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=2,
    interceptions=0,
    ball_recoveries=1,
    dribbled_past=0,
    duels_won=2,
    duels_lost=0,
    ground_duels_won=2,
    ground_duels_total=2,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=1,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Saka's match rating: 8.00


In [106]:
player = PlayerProfile("Rogers", "England", "midfielder", 83)

player.input_match_stats(
    minutes=18,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=5,
    total_passes=7,
    expected_goals=0,
    expected_assists=0.3,
    successful_dribbles=0,
    total_dribbles=1,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=0,
    tackles_won=1,
    tackles_loss=0,
    clearances=1,
    interceptions=0,
    ball_recoveries=2,
    dribbled_past=0,
    duels_won=3,
    duels_lost=1,
    ground_duels_won=2,
    ground_duels_total=3,
    aerial_duels_won=1,
    aerial_duels_total=1,
    fouled=1,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Rogers's match rating: 6.70


In [108]:
player = PlayerProfile("Spence", "England", "defender", 79)

player.input_match_stats(
    minutes=10,
    goals=0,
    assists=0,
    total_shots=1,
    shots_on_target=1,
    accurate_passes=5,
    total_passes=6,
    expected_goals=0.56,
    expected_assists=0,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=1,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=0,
    dribbled_past=0,
    duels_won=1,
    duels_lost=0,
    ground_duels_won=1,
    ground_duels_total=1,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Spence's match rating: 6.94


In [110]:
player = PlayerProfile("Guehi", "England", "defender", 85)

player.input_match_stats(
    minutes=3,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=7,
    total_passes=7,
    expected_goals=0,
    expected_assists=0,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=1,
    total_long_balls=1,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=1,
    dribbled_past=0,
    duels_won=0,
    duels_lost=0,
    ground_duels_won=0,
    ground_duels_total=0,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Guehi's match rating: 6.50


In [118]:
#AwayTeam
awayteam_world_cup_starting_xi_vs_hometeam = [
    ["Dominik Livakovic", "GK", "Dinamo Zagreb", 80],
    ["Josko Gvardiol", "CB", "Manchester City", 83],
    ["Josip Sutalo", "CB", "Ajax Amsterdam", 79],
    ["Josip Stanisic", "RWB", "Bayern Munich", 82],
    ["Luka Vuskovic", "CB", "Hamburg", 81],
    ["Ivan Perisic", "LWB", "PSV Eindhoven", 79],
    ["Mario Pasalic", "CAM", "Atalanta", 78],
    ["Luka Modric", "CM", "AC Milan", 84],
    ["Martin Baturina", "CAM", "Como", 79],
    ["Petar Sucic", "CM", "Inter Milan", 79],
    ["Petar Musa", "ST", "Dallas", 78],

]




awayteam_world_cup_bench_vs_hometeam = [
    ["Mateo Kovacic", "CM", "Manchester City", 80],
    ["Marco Pasalic", "RWB", "Orlando City", 75],
    ["Igor Matanovic", "ST", "Freiburg", 76],
    ["Andrej Kramaric", "CAM", "Hoffenheim", 81],
    ["Nikola Vlasic", "CAM", "Torino", 79],

]   

In [120]:
gk = GoalkeeperProfile("Livakovic", "Croatia", "goalkeeper", 80)
gk.input_match_stats(
    minutes_played=90,
    saves=7,
    saves_inside_box=7,
    saves_outside_box=0,
    goals_conceded=4,
    xG_faced=3.33,
    goals_prevented=-0.67,
    total_passes=35,
    accurate_passes=24,
    total_long_balls=20,
    accurate_long_balls=9,
    touches=49,
    errors=0,
    penalty_saves=0,
    clean_sheet=False,
    yellow_card=False
)
print(f"{gk.name}'s match rating: {gk.match_rating:.2f}")

Livakovic's match rating: 8.12


In [122]:
player = PlayerProfile("Gvardiol", "Croatia", "defender", 83)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=0,
    total_shots=1,
    shots_on_target=0,
    accurate_passes=43,
    total_passes=47,
    expected_goals=0.2,
    expected_assists=0,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=1,
    total_long_balls=3,
    dispossessed=0,
    tackles_won=1,
    tackles_loss=0,
    clearances=1,
    interceptions=0,
    ball_recoveries=4,
    dribbled_past=1,
    duels_won=2,
    duels_lost=2,
    ground_duels_won=1,
    ground_duels_total=3,
    aerial_duels_won=1,
    aerial_duels_total=1,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=4 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Gvardiol's match rating: 5.50


In [124]:
player = PlayerProfile("Vuskovic", "Croatia", "defender", 81)

player.input_match_stats(
    minutes=66,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=37,
    total_passes=42,
    expected_goals=0,
    expected_assists=0.01,
    successful_dribbles=1,
    total_dribbles=2,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=3,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=5,
    interceptions=1,
    ball_recoveries=6,
    dribbled_past=0,
    duels_won=2,
    duels_lost=2,
    ground_duels_won=2,
    ground_duels_total=4,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=1,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=4 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Vuskovic's match rating: 5.67


In [126]:
player = PlayerProfile("Sutalo", "Croatia", "defender", 79)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=0,
    total_shots=1,
    shots_on_target=0,
    accurate_passes=63,
    total_passes=70,
    expected_goals=0.17,
    expected_assists=0.01,
    successful_dribbles=1,
    total_dribbles=1,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=1,
    total_long_balls=6,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=4,
    interceptions=2,
    ball_recoveries=3,
    dribbled_past=1,
    duels_won=0,
    duels_lost=0,
    ground_duels_won=1,
    ground_duels_total=2,
    aerial_duels_won=4,
    aerial_duels_total=4,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=4 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Sutalo's match rating: 5.97


In [128]:
player = PlayerProfile("Stanisic", "Croatia", "defender", 82)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=30,
    total_passes=34,
    expected_goals=0,
    expected_assists=0.03,
    successful_dribbles=1,
    total_dribbles=1,
    accurate_crosses=1,
    total_crosses=1,
    accurate_long_balls=1,
    total_long_balls=2,
    dispossessed=0,
    tackles_won=2,
    tackles_loss=0,
    clearances=1,
    interceptions=1,
    ball_recoveries=6,
    dribbled_past=1,
    duels_won=4,
    duels_lost=2,
    ground_duels_won=4,
    ground_duels_total=5,
    aerial_duels_won=0,
    aerial_duels_total=1,
    fouled=1,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=4 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Stanisic's match rating: 7.62


In [130]:
player = PlayerProfile("Modric", "Croatia", "midfielder", 84)

player.input_match_stats(
    minutes=58,
    conceded_penalty=1,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=27,
    total_passes=27,
    expected_goals=0,
    expected_assists=0.01,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=1,
    accurate_long_balls=1,
    total_long_balls=1,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=1,
    ball_recoveries=1,
    dribbled_past=0,
    duels_won=1,
    duels_lost=1,
    ground_duels_won=1,
    ground_duels_total=2,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=1,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Modric's match rating: 4.25


In [132]:
player = PlayerProfile("Pasalic", "Croatia", "midfielder", 78)

player.input_match_stats(
    minutes=78,
    goals=0,
    assists=0,
    total_shots=1,
    shots_on_target=0,
    accurate_passes=21,
    total_passes=22,
    expected_goals=0.02,
    expected_assists=0.42,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=1,
    total_long_balls=1,
    dispossessed=0,
    tackles_won=1,
    tackles_loss=0,
    clearances=1,
    interceptions=0,
    ball_recoveries=2,
    dribbled_past=1,
    duels_won=1,
    duels_lost=2,
    ground_duels_won=1,
    ground_duels_total=3,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=0,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Pasalic's match rating: 5.55


In [134]:
player = PlayerProfile("Perisic", "Croatia", "midfielder", 79)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=1,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=25,
    total_passes=29,
    expected_goals=0,
    expected_assists=0.17,
    successful_dribbles=0,
    total_dribbles=1,
    accurate_crosses=1,
    total_crosses=6,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=0,
    tackles_won=1,
    tackles_loss=0,
    clearances=2,
    interceptions=2,
    ball_recoveries=2,
    dribbled_past=0,
    duels_won=5,
    duels_lost=5,
    ground_duels_won=1,
    ground_duels_total=4,
    aerial_duels_won=4,
    aerial_duels_total=6,
    fouled=0,
    number_of_fouls=2,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Perisic's match rating: 8.44


In [136]:
player = PlayerProfile("Sucic", "Croatia", "midfielder", 79)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=1,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=35,
    total_passes=42,
    expected_goals=0.01,
    expected_assists=0.03,
    successful_dribbles=1,
    total_dribbles=2,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=1,
    total_long_balls=2,
    dispossessed=2,
    tackles_won=1,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=5,
    dribbled_past=3,
    duels_won=3,
    duels_lost=8,
    ground_duels_won=3,
    ground_duels_total=11,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=1,
    number_of_fouls=2,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Sucic's match rating: 8.14


In [138]:
player = PlayerProfile("Baturina", "Croatia", "midfielde", 79)

player.input_match_stats(
    minutes=78,
    goals=1,
    assists=0,
    total_shots=1,
    shots_on_target=1,
    accurate_passes=25,
    total_passes=32,
    expected_goals=0.04,
    expected_assists=0.05,
    successful_dribbles=1,
    total_dribbles=2,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=2,
    total_long_balls=3,
    dispossessed=0,
    tackles_won=1,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=3,
    dribbled_past=0,
    duels_won=6,
    duels_lost=3,
    ground_duels_won=4,
    ground_duels_total=5,
    aerial_duels_won=2,
    aerial_duels_total=4,
    fouled=2,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Baturina's match rating: 9.26


In [140]:
player = PlayerProfile("Musa", "Croatia", "forard", 78)

player.input_match_stats(
    minutes=66,
    goals=1,
    assists=0,
    total_shots=1,
    shots_on_target=1,
    accurate_passes=3,
    total_passes=8,
    expected_goals=0.14,
    expected_assists=0,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=0,
    tackles_won=1,
    tackles_loss=0,
    clearances=1,
    interceptions=0,
    ball_recoveries=0,
    dribbled_past=0,
    duels_won=5,
    duels_lost=7,
    ground_duels_won=2,
    ground_duels_total=3,
    aerial_duels_won=3,
    aerial_duels_total=9,
    fouled=1,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Musa's match rating: 7.52


In [142]:
player = PlayerProfile("Kovacic", "Croatia", "midfielder", 80)

player.input_match_stats(
    minutes=32,
    goals=0,
    assists=0,
    total_shots=1,
    shots_on_target=1,
    accurate_passes=30,
    total_passes=30,
    expected_goals=0.04,
    expected_assists=0.02,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=1,
    total_long_balls=1,
    dispossessed=0,
    tackles_won=1,
    tackles_loss=0,
    clearances=0,
    interceptions=1,
    ball_recoveries=1,
    dribbled_past=0,
    duels_won=3,
    duels_lost=1,
    ground_duels_won=3,
    ground_duels_total=4,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=2,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Kovacic's match rating: 7.17


In [144]:
player = PlayerProfile("Pasalic", "Croatia", "midfielder", 76)

player.input_match_stats(
    minutes=24,
    goals=0,
    assists=0,
    total_shots=1,
    shots_on_target=1,
    accurate_passes=6,
    total_passes=10,
    expected_goals=0.06,
    expected_assists=0.01,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=2,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=2,
    tackles_won=1,
    tackles_loss=0,
    clearances=1,
    interceptions=0,
    ball_recoveries=2,
    dribbled_past=0,
    duels_won=1,
    duels_lost=2,
    ground_duels_won=1,
    ground_duels_total=3,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Pasalic's match rating: 5.80


In [146]:
player = PlayerProfile("Matanovic", "Croatia", "forward", 76)

player.input_match_stats(
    minutes=24,
    goals=0,
    assists=0,
    total_shots=1,
    shots_on_target=1,
    accurate_passes=0,
    total_passes=0,
    expected_goals=0.01,
    expected_assists=0,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=0,
    dribbled_past=0,
    duels_won=2,
    duels_lost=2,
    ground_duels_won=1,
    ground_duels_total=2,
    aerial_duels_won=1,
    aerial_duels_total=2,
    fouled=1,
    number_of_fouls=2,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Matanovic's match rating: 6.05


In [148]:
player = PlayerProfile("Kramaric", "Croatia", "forward", 80)

player.input_match_stats(
    minutes=12,
    goals=0,
    assists=0,
    total_shots=1,
    shots_on_target=0,
    accurate_passes=10,
    total_passes=10,
    expected_goals=0.01,
    expected_assists=0.01,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=2,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=1,
    dribbled_past=0,
    duels_won=0,
    duels_lost=0,
    ground_duels_won=0,
    ground_duels_total=0,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Kramaric's match rating: 5.65


In [150]:
player = PlayerProfile("Vlasic", "Croatia", "forward", 79)

player.input_match_stats(
    minutes=12,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=2,
    total_passes=4,
    expected_goals=0,
    expected_assists=0,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=1,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=1,
    ball_recoveries=0,
    dribbled_past=0,
    duels_won=0,
    duels_lost=1,
    ground_duels_won=0,
    ground_duels_total=1,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=0,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Vlasic's match rating: 5.73
